In [1]:
import modules.data as d
import modules.utils as u
from pathlib import Path

#### init ####
dataset_dir = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')
device = u.Devices().auto_set_device()#drop=['cuda:4','cuda:7'])

('cuda:4', 'NVIDIA A100-SXM4-80GB', 81043)
('cuda:2', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:3', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:6', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:7', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:0', 'NVIDIA A100-SXM4-80GB', 78513)
('cuda:1', 'NVIDIA A100-SXM4-80GB', 50693)
('cuda:5', 'NVIDIA A100-SXM4-80GB', 30193)

# #### Device() ####
# device = cuda:4



In [2]:
#### data ####
brca = d.TCGA(
    tcga_project = 'BRCA',
    tcga_dir = dataset_dir/'tcga',
    # type_col = 'sample_type',
    subtype_col = 'paper_BRCA_Subtype_PAM50',
    drop = ['Normal', 'Primary Tumor', 'Metastatic'],
    gene_name_path = dataset_dir/'other'/'name2ensg.csv',
    keep_noname = False,
)

kegg = d.KEGG(
    relation_filepath=dataset_dir/'other'/'relation_ohe.csv', 
    counts_data=brca,
)

dataset = d.GraphDataset(brca, kegg, kegg)
# _batch = d.get_toy_databatch(dataset, device)

# #### KEGG() ####
# _orig_kwargs             5                        dict
# relation                 (75939, 19)              DataFrame
# ensg                     4373                     list
# pathway_labels           305                      list
# edge_index               (2, 32464)               Tensor (cuda:4)
# edge_attr                (32464, 16)              Tensor (cuda:4)
# edge_labels              16                       list
# pathway_index            (4373, 305)              Tensor (cuda:4)

# #### TCGA() ####
# _orig_kwargs             9                        dict
# counts_path              PosixPath
# metadata_path            PosixPath
# gene_name_path           PosixPath
# metadata_complete        (1231, 93)               DataFrame
# metadata                 (1172, 2)                DataFrame
# y                        (1172,)                  Tensor (cuda:4)
# y_labels                 5                        list
# ensgv                    (60660, 3)             

---

1a: dimensions.

* scVI tested at the following (for 5 trials):
    * 4e-4, 1 layer, 128 neurons
    * 1e-3, 1-3 layers, 128 or 256 neurons
    * 5e-4, 1 layer, 128 neurons

* 1a will focus on replicating these results quickly (preliminary) so I can start other experiments. Can add more afterwards.

* Variables:
    * embed_dim = [128, 256]
    * hidden_dims = [1, 2, 3]
    * lr = [5e-4, 1e-3]

* 12 configs, max 300 epochs.

In [3]:
from modules.layers import AttentionSetPooling
from modules.model import BaseAutoencoder
from modules.norm import LogCounts
from modules.train import Experiment, grid
from modules.trainers import ReconstrTrainer
from functools import partial
import torch.nn as nn

In [4]:
## Model

model_template = partial(
    BaseAutoencoder,
    dataset = dataset,
    # out_dim = num_nodes (default)
    # embed_dim = 128, 256 (grid) ***
    # head_dim = None (default)
    # num_heads = 1 (default)
    method = 'node',

    # layers
    norm_class=LogCounts, 
    encoder_class=nn.Linear,
    pooling_class=AttentionSetPooling,
    mlp=False,
    variational=False,
    # out_module = nn.Linear (default)

    # layer params
    # hidden_dims = [128, 64] (grid) ***
    act_fn=nn.ReLU, 
    norm_fn='layer', 
    end_fn=False,

    # kwargs
    norm_kwargs = {'libnorm':True, 'znorm':True, 'learnable':True},
    # pooling_kwargs = None (default)
)
 
model_grid = grid(
    model_template,
    embed_dim=[16,32,64,128,256],
    hidden_dims=[0,1,2,3,4],
)

## Trainer

trainer_template = partial(
    ReconstrTrainer,
    # lr = [5e-4, 1e-3] (grid) ***
    # pos_keys = None (default)
    # out_keys = {'input':'x_t_pred', 'target':'x_t'} (default)
    # batch_keys = {'x':'x'} (default)
    # loss_class = nn.MSELoss (default)
    # loss_kwargs = None (default)
    # optim_class = torch.optim.Adam (default)
    # optim_kwargs = None (default)
    early_stop = True,
    stop_metric = 'mae',
    # stop_kwargs = None (default)
    trainer_norm_class=LogCounts,
    # trainer_norm_kwargs = None (default)

)

trainer_grid = grid(
    trainer_template,
    # lr=[1e-4, 5e-4, 1e-3, 5e-3], # run #1
    lr=[1e-2] # run #2
)

## Experiment

expt = Experiment(
    num_trials=5, # scVI: 5 trials
    num_epochs=300, # scVI: 200 epochs
    dataset=dataset,
    device=device,
    batch_size=128
)

expt.add_grid(
    model_grid=model_grid,
    trainer_grid=trainer_grid
)

print(len(expt.configs))

25


In [5]:
expt.run_experiment(
    'grid_1a2_reconstr_dims_xtra',
    report_metrics=['loss','rmse','mae','r2'],
    save_csv=True,
    save_params=True,
    save_values=True,
    verbose=False,
)

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

...

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]